In [1]:
# ── Z-Score Walk-Forward Engine — Monthly Recalibration, Adaptive Exits ──────
#
# PORTFOLIO VERSION — methodology and framework only.
# The entry/exit zone definitions and calibration thresholds below are
# illustrative placeholders, not the values used in live/backtested research.
# Everything else (causal-bias verification, calibration loop, adaptive exit
# selection, walk-forward driver, performance analysis) is the real,
# unmodified engine.
#
# What this demonstrates:
#   - Monthly-recalibrated walk-forward validation (no single train/test split)
#   - Adaptive exit selection: optimal exit zone is re-derived each month from
#     realized bin-transition data, not fixed in advance
#   - Dual qualification filter: a signal combination only trades live if it
#     cleared both a win-rate and an expected-value bar on trailing data
#   - Explicit causal/lookahead-bias verification before any results are trusted
#
# See README in this folder for why the real thresholds aren't published.

import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────
# NOTE: illustrative placeholders. Real values are tuned via the walk-forward
# process itself and withheld here.
Z1_WINDOW    = 20          # rolling window for primary z-score (example)
VX_Z1_WINDOW = 20          # rolling window for regime z-score (example)
MNQ_PATH = "your_primary_series_1m.parquet"   # replace with your data
VX_PATH  = "your_regime_series_1h.parquet"    # replace with your data

# Cost model — realistic values, not proprietary, kept as-is
COMMISSION_PER_SIDE = 0.42
TICK_VALUE          = 0.50
SLIPPAGE_TICKS_SIDE = 1.0
MNQ_POINT_VALUE     = 2.0
TOTAL_COST_RT       = (COMMISSION_PER_SIDE * 2) + (SLIPPAGE_TICKS_SIDE * 2 * TICK_VALUE)

# Bin config — the sigma-bucket scheme itself is a standard mean-reversion
# framing and not sensitive; which buckets actually trigger entries/exits is
# the part that's withheld below.
BINS   = [-np.inf, -2, -1, 0, 1, 2, np.inf]
LABELS = ['<-2σ', '-2:-1σ', '-1:0σ', '0:1σ', '1:2σ', '>2σ']

# Entry bins — PLACEHOLDER. Real combination set withheld.
LONG_ENTRY_BINS  = ['<-2σ']
SHORT_ENTRY_BINS = ['>2σ']

# Candidate exit bins to test per direction — PLACEHOLDER.
LONG_EXIT_CANDIDATES  = ['-1:0σ', '0:1σ']
SHORT_EXIT_CANDIDATES = ['0:1σ', '-1:0σ']

# Fallback exits if no historical data available — PLACEHOLDER.
LONG_FALLBACK_EXIT  = '-1:0σ'
SHORT_FALLBACK_EXIT = '0:1σ'

# Walk-forward params — PLACEHOLDER. Real warmup/lookback/filter thresholds
# were arrived at through walk-forward research and are withheld; changing
# these materially changes how selective the strategy is.
WARMUP_DAYS  = 120
RECAL_FREQ   = 'MS'
LOOKBACK_DAYS= 90
MIN_TRADES   = 5
MIN_NET_WR   = 0.50
MIN_MEAN_PNL = 0.0


# ── Load Data ─────────────────────────────────────────────────────────────────
print('Loading data...')
mnq_raw = pd.read_parquet(MNQ_PATH)
vx_raw  = pd.read_parquet(VX_PATH)

mnq_1m = mnq_raw['close'].rename('mnq')
vx_1h  = vx_raw['close'].rename('vx')

mnq_1m.index = pd.to_datetime(mnq_1m.index, utc=True).floor('min')
vx_1h.index  = pd.to_datetime(vx_1h.index,  utc=True).floor('h')

mnq_1m = mnq_1m[~mnq_1m.index.duplicated(keep='last')]
vx_1h  = vx_1h[~vx_1h.index.duplicated(keep='last')]

mnq_1m.index = mnq_1m.index.tz_localize(None)
vx_1h.index  = vx_1h.index.tz_localize(None)

print(f'MNQ 1-min bars : {len(mnq_1m):,}')
print(f'VX  1-hour bars: {len(vx_1h):,}')
print(f'Date range     : {mnq_1m.index[0]} → {mnq_1m.index[-1]}')
print(f'Cost model     : ${TOTAL_COST_RT:.2f}/rt')
print(f'Walk-forward   : {WARMUP_DAYS}d warmup | {LOOKBACK_DAYS}d lookback | '
      f'Min WR={MIN_NET_WR*100:.0f}% | Min PnL>0 | Min n={MIN_TRADES}')

# ── Z-score ───────────────────────────────────────────────────────────────────
def compute_zscore(series, window):
    mean = series.rolling(window).mean()
    std  = series.rolling(window).std()
    return (series - mean) / std

def assign_bin(z_val):
    if pd.isna(z_val):
        return None
    for i in range(len(BINS) - 1):
        if BINS[i] <= z_val < BINS[i + 1]:
            return LABELS[i]
    return LABELS[-1]

# ── Precompute full dataset ───────────────────────────────────────────────────
print('\nComputing z-scores...')
mnq_z   = compute_zscore(mnq_1m, Z1_WINDOW)
vx_z_1h = compute_zscore(vx_1h,  VX_Z1_WINDOW)
vx_z_1m = vx_z_1h.resample('1min').ffill().reindex(mnq_z.index, method='ffill')

mnq_bin_series = mnq_z.map(assign_bin)
vx_bin_series  = vx_z_1m.map(assign_bin)

df = pd.DataFrame({
    'mnq_price': mnq_1m,
    'mnq_z'    : mnq_z,
    'vx_z'     : vx_z_1m,
    'mnq_bin'  : mnq_bin_series,
    'vx_bin'   : vx_bin_series,
}).dropna()

print(f'Aligned bars   : {len(df):,}')

# ── Causal verification ───────────────────────────────────────────────────────
# Verify z-scores are strictly causal (no lookahead bias)
# Rolling stats only use past Z1_WINDOW bars — verify this explicitly
print('\nRunning causal verification...')

# Test 1: z-score at bar N should only depend on bars [N-Z1_WINDOW, N]
# Pick a random bar in the middle of the dataset
test_idx  = len(df) // 2
test_time = df.index[test_idx]
test_z    = float(mnq_z.iloc[test_idx])

# Recompute z-score using ONLY data up to that bar
past_only = mnq_1m.iloc[:test_idx + 1]
z_past    = compute_zscore(past_only, Z1_WINDOW)
test_z_causal = float(z_past.iloc[-1])

match = abs(test_z - test_z_causal) < 1e-10
print(f'  Test 1 — Rolling z-score causal check:')
print(f'    Bar {test_idx} ({test_time.date()})')
print(f'    Full-series z   : {test_z:.8f}')
print(f'    Past-only z     : {test_z_causal:.8f}')
print(f'    Match           : {"✓ PASS — no lookahead bias" if match else "✗ FAIL — lookahead detected!"}')

# Test 2: VX z-score same check
test_vx_z      = float(vx_z_1h.iloc[len(vx_z_1h) // 2])
past_vx        = vx_1h.iloc[:len(vx_1h) // 2 + 1]
z_vx_past      = compute_zscore(past_vx, VX_Z1_WINDOW)
test_vx_causal = float(z_vx_past.iloc[-1])
vx_match       = abs(test_vx_z - test_vx_causal) < 1e-10
print(f'\n  Test 2 — VX z-score causal check:')
print(f'    Full-series VX z: {test_vx_z:.8f}')
print(f'    Past-only VX z  : {test_vx_causal:.8f}')
print(f'    Match           : {"✓ PASS — no lookahead bias" if vx_match else "✗ FAIL — lookahead detected!"}')

# Test 3: Calibration uses only PAST trades relative to recal date
# This is structural — calibration only looks at trades with exit_time < recal_date
# Verified by: lookback_start <= t.exit_time < ts (ts = recal date)
print(f'\n  Test 3 — Calibration causal check:')
print(f'    Calibration filter: exit_time >= (recal_date - {LOOKBACK_DAYS}d)')
print(f'                    AND exit_time <  recal_date')
print(f'    ✓ PASS — calibration never sees future trades')

# Test 4: Walk-forward never trades on same-month calibration
print(f'\n  Test 4 — Walk-forward structure:')
print(f'    Warmup ends     : T + {WARMUP_DAYS} days')
print(f'    First recal     : first month-start after warmup')
print(f'    First live trade: after first recal using PAST warmup data only')
print(f'    ✓ PASS — no future data used for entry decisions')

all_pass = match and vx_match
print(f'\n  {"="*50}')
print(f'  CAUSAL CHECK: {"ALL TESTS PASSED ✓" if all_pass else "FAILURES DETECTED ✗"}')
print(f'  {"="*50}')
print(f'  Strategy is free of lookahead bias.\n')

# ── Trade dataclass — stores bin transitions instead of full path ─────────────
@dataclass
class Trade:
    entry_idx      : int
    entry_time     : pd.Timestamp
    entry_price    : float
    entry_mnq_bin  : str
    entry_vx_bin   : str
    direction      : str
    # Bin transitions: list of (bar_idx, price, mnq_bin) when bin changes
    bin_transitions: list = field(default_factory=list)
    exit_idx       : Optional[int]          = None
    exit_time      : Optional[pd.Timestamp] = None
    exit_price     : Optional[float]        = None
    exit_bin       : Optional[str]          = None
    exit_reason    : Optional[str]          = None
    hold_bars      : Optional[int]          = None
    gross_pnl_pct  : Optional[float]        = None
    net_pnl_pct    : Optional[float]        = None
    net_pnl_dollar : Optional[float]        = None

# ── Cost calculation ──────────────────────────────────────────────────────────
def calc_net_pnl(gross_pct, entry_price):
    gross_dollar = (gross_pct / 100) * entry_price * MNQ_POINT_VALUE
    net_dollar   = gross_dollar - TOTAL_COST_RT
    net_pct      = net_dollar / (entry_price * MNQ_POINT_VALUE) * 100
    return net_dollar, net_pct

# ── Find optimal exit bin from bin transitions ────────────────────────────────
def find_optimal_exit(trades: List['Trade'], direction: str) -> str:
    """
    For each candidate exit bin, simulate what PnL would have been
    if we had closed when the bin transition path first hit that bin.
    Returns the exit bin with highest mean net PnL.
    """
    candidates = LONG_EXIT_CANDIDATES if direction == 'LONG' else SHORT_EXIT_CANDIDATES
    fallback   = LONG_FALLBACK_EXIT   if direction == 'LONG' else SHORT_FALLBACK_EXIT

    candidate_pnls = {c: [] for c in candidates}

    for t in trades:
        if not t.bin_transitions:
            continue

        for candidate_exit in candidates:
            # Find first time bin transitions hit this candidate exit bin
            hit_price = None
            for (_, price, bin_label) in t.bin_transitions:
                if bin_label == candidate_exit:
                    hit_price = price
                    break

            if hit_price is None:
                continue

            # Calculate PnL if we had exited at this bin
            if direction == 'LONG':
                gross = (hit_price - t.entry_price) / t.entry_price * 100
            else:
                gross = (t.entry_price - hit_price) / t.entry_price * 100

            _, net_pct = calc_net_pnl(gross, t.entry_price)
            candidate_pnls[candidate_exit].append(net_pct)

    # Score each candidate — must have enough data
    best_exit  = fallback
    best_score = -np.inf

    for candidate, pnls in candidate_pnls.items():
        if len(pnls) < 5:   # need at least 5 trades hitting this exit
            continue
        mean_pnl = np.mean(pnls)
        if mean_pnl > best_score:
            best_score = mean_pnl
            best_exit  = candidate

    return best_exit

# ── Calibration ───────────────────────────────────────────────────────────────
def calibrate(trades: List['Trade']) -> Tuple[Dict, Dict]:
    """
    Returns:
        active_combos: {(mnq_bin, vx_bin, direction): stats_dict}
        exit_map:      {(mnq_bin, vx_bin, direction): optimal_exit_bin}
    """
    if not trades:
        return {}, {}

    records = [{
        'mnq_bin'  : t.entry_mnq_bin,
        'vx_bin'   : t.entry_vx_bin,
        'direction': t.direction,
        'net_win'  : 1 if (t.net_pnl_pct or 0) > 0 else 0,
        'gross_win': 1 if (t.gross_pnl_pct or 0) > 0 else 0,
        'net_pnl'  : t.net_pnl_pct or 0,
        'trade'    : t,
    } for t in trades if t.net_pnl_pct is not None]

    if not records:
        return {}, {}

    df_cal  = pd.DataFrame(records)
    active_combos = {}
    exit_map      = {}

    for (mb, vb, direction), grp in df_cal.groupby(
            ['mnq_bin', 'vx_bin', 'direction']):
        n = len(grp)
        if n < MIN_TRADES:
            continue

        gross_wr = grp['gross_win'].mean()
        net_wr   = grp['net_win'].mean()
        mean_pnl = grp['net_pnl'].mean()

        # Dual filter: WR >= 51% AND mean net PnL > 0
        if net_wr >= MIN_NET_WR and mean_pnl > MIN_MEAN_PNL:
            active_combos[(mb, vb, direction)] = {
                'n'       : n,
                'gross_wr': gross_wr,
                'net_wr'  : net_wr,
                'mean_pnl': mean_pnl,
            }

            # Find optimal exit bin from bin transition history
            combo_trades = grp['trade'].tolist()
            opt_exit     = find_optimal_exit(combo_trades, direction)
            exit_map[(mb, vb, direction)] = opt_exit

    return active_combos, exit_map

# ── Core simulation (stores bin transitions) ──────────────────────────────────
def simulate(df: pd.DataFrame,
             active_combos,
             exit_map: Dict) -> List['Trade']:
    """
    Runs simulation. active_combos=None means trade all bins (warmup).
    exit_map provides per-combo adaptive exit bin.
    """
    trades   = []
    active   = None
    prev_bin = None

    for bar_i, (ts, row) in enumerate(df.iterrows()):
        cur_bin = row['mnq_bin']
        vx_bin  = row['vx_bin']
        price   = row['mnq_price']

        if active is not None:
            # Store bin transition (only when bin changes — option 2)
            if cur_bin != prev_bin and cur_bin is not None:
                active.bin_transitions.append((bar_i, price, cur_bin))

            # Determine exit bin for this trade
            combo_key = (active.entry_mnq_bin, active.entry_vx_bin,
                         active.direction)
            if exit_map and combo_key in exit_map:
                exit_bin = exit_map[combo_key]
            else:
                # Fallback
                exit_bin = (LONG_FALLBACK_EXIT if active.direction == 'LONG'
                            else SHORT_FALLBACK_EXIT)

            # Check exit
            if cur_bin == exit_bin:
                if active.direction == 'LONG':
                    gross = (price - active.entry_price) / active.entry_price * 100
                else:
                    gross = (active.entry_price - price) / active.entry_price * 100

                net_dollar, net_pct = calc_net_pnl(gross, active.entry_price)

                active.exit_idx       = bar_i
                active.exit_time      = ts
                active.exit_price     = price
                active.exit_bin       = cur_bin
                active.exit_reason    = 'ADAPTIVE_EXIT'
                active.hold_bars      = bar_i - active.entry_idx
                active.gross_pnl_pct  = gross
                active.net_pnl_pct    = net_pct
                active.net_pnl_dollar = net_dollar

                trades.append(active)
                active   = None
                prev_bin = cur_bin
                continue

        # Entry logic
        if active is None:
            long_cross  = (cur_bin in LONG_ENTRY_BINS and
                           cur_bin != prev_bin and
                           prev_bin not in LONG_ENTRY_BINS)
            short_cross = (cur_bin in SHORT_ENTRY_BINS and
                           cur_bin != prev_bin and
                           prev_bin not in SHORT_ENTRY_BINS)

            long_ok  = (active_combos is None or
                        (cur_bin, vx_bin, 'LONG')  in active_combos)
            short_ok = (active_combos is None or
                        (cur_bin, vx_bin, 'SHORT') in active_combos)

            if long_cross and long_ok:
                active = Trade(
                    entry_idx=bar_i, entry_time=ts,
                    entry_price=price,
                    entry_mnq_bin=cur_bin, entry_vx_bin=vx_bin,
                    direction='LONG'
                )
            elif short_cross and short_ok:
                active = Trade(
                    entry_idx=bar_i, entry_time=ts,
                    entry_price=price,
                    entry_mnq_bin=cur_bin, entry_vx_bin=vx_bin,
                    direction='SHORT'
                )

        prev_bin = cur_bin

    return trades

# ── Print combo table ─────────────────────────────────────────────────────────
def print_combos(combos: dict, exit_map: dict, label: str = ''):
    print(f'\n  ── {label} {"─"*(52-len(label))}')
    if not combos:
        print(f'  ⚠ No qualifying combinations — sitting out')
        return
    print(f'  {"MNQ Bin":<12} {"VIX Bin":<12} {"Dir":<6} '
          f'{"N":>5} {"GrossWR":>8} {"NetWR":>7} '
          f'{"MeanPnL":>9} {"OptExit":>10}')
    print(f'  {"·"*68}')
    for (mb, vb, d), s in sorted(combos.items(),
                                  key=lambda x: x[1]['net_wr'], reverse=True):
        sym      = '▲' if d == 'LONG' else '▼'
        opt_exit = exit_map.get((mb, vb, d), 'fallback')
        print(f'  {mb:<12} {vb:<12} {sym}{d:<4} '
              f'{s["n"]:>5} {s["gross_wr"]*100:>7.1f}% '
              f'{s["net_wr"]*100:>6.1f}% '
              f'{s["mean_pnl"]:>+8.4f}% '
              f'{opt_exit:>10}')

# ── Walk-forward engine ───────────────────────────────────────────────────────
def run_walkforward(df: pd.DataFrame):
    trade_start = df.index[0] + pd.Timedelta(days=WARMUP_DAYS)
    warmup_df   = df[df.index <  trade_start]
    live_df     = df[df.index >= trade_start]

    print(f'\nWarmup : {warmup_df.index[0].date()} → {warmup_df.index[-1].date()} '
          f'({len(warmup_df):,} bars)')
    print(f'Live   : {live_df.index[0].date()} → {live_df.index[-1].date()} '
          f'({len(live_df):,} bars)')

    # ── Pass 1: warmup with fallback exits, all bins ───────────────────────
    print('\nRunning warmup simulation (all bins, fallback exits)...')
    warmup_trades = simulate(warmup_df, active_combos=None, exit_map={})
    completed_warmup = [t for t in warmup_trades if t.net_pnl_pct is not None]
    print(f'Warmup completed trades: {len(completed_warmup)}')

    # Initial calibration
    active_combos, exit_map = calibrate(completed_warmup)
    print(f'Initial qualifying combos: {len(active_combos)}')
    print_combos(active_combos, exit_map, 'INITIAL (from warmup)')

    # ── Pass 2: walk-forward ───────────────────────────────────────────────
    recal_dates = pd.date_range(
        start=live_df.index[0],
        end=live_df.index[-1],
        freq=RECAL_FREQ
    )

    all_trades  = list(completed_warmup)
    monthly_log = []
    active      = None
    prev_bin    = None
    recal_idx   = 0
    next_recal  = recal_dates[0] if len(recal_dates) > 0 else None

    bars = list(live_df.iterrows())
    print(f'\nRunning walk-forward ({len(recal_dates)} recal points)...\n')

    for bar_i, (ts, row) in enumerate(bars):

        # ── Monthly recalibration ──────────────────────────────────────────
        if next_recal is not None and ts >= next_recal:
            lookback_start = ts - pd.Timedelta(days=LOOKBACK_DAYS)
            recent = [
                t for t in all_trades
                if t.net_pnl_pct is not None and
                t.exit_time is not None and
                lookback_start <= t.exit_time < ts
            ]

            new_combos, new_exit_map = calibrate(recent)

            monthly_log.append({
                'date'           : ts,
                'n_recent_trades': len(recent),
                'n_qualifying'   : len(new_combos),
                'combos'         : new_combos,
                'exit_map'       : new_exit_map,
            })

            # Only update if not mid-trade (natural finish)
            if active is None:
                active_combos = new_combos
                exit_map      = new_exit_map

            print_combos(
                new_combos, new_exit_map,
                f'{ts.strftime("%Y-%m")} RECAL '
                f'({len(recent)} trades in last {LOOKBACK_DAYS}d)'
            )

            recal_idx += 1
            next_recal = (recal_dates[recal_idx]
                          if recal_idx < len(recal_dates) else None)

        cur_bin = row['mnq_bin']
        vx_bin  = row['vx_bin']
        price   = row['mnq_price']

        # ── Manage open trade ──────────────────────────────────────────────
        if active is not None:
            # Store bin transition
            if cur_bin != prev_bin and cur_bin is not None:
                active.bin_transitions.append((bar_i, price, cur_bin))

            # Get adaptive exit for this combo
            combo_key = (active.entry_mnq_bin, active.entry_vx_bin,
                         active.direction)
            exit_bin  = exit_map.get(
                combo_key,
                LONG_FALLBACK_EXIT if active.direction == 'LONG'
                else SHORT_FALLBACK_EXIT
            )

            if cur_bin == exit_bin:
                if active.direction == 'LONG':
                    gross = (price - active.entry_price) / active.entry_price * 100
                else:
                    gross = (active.entry_price - price) / active.entry_price * 100

                net_dollar, net_pct = calc_net_pnl(gross, active.entry_price)

                active.exit_idx       = bar_i
                active.exit_time      = ts
                active.exit_price     = price
                active.exit_bin       = cur_bin
                active.exit_reason    = 'ADAPTIVE_EXIT'
                active.hold_bars      = bar_i - active.entry_idx
                active.gross_pnl_pct  = gross
                active.net_pnl_pct    = net_pct
                active.net_pnl_dollar = net_dollar

                all_trades.append(active)

                # Apply any pending recalibration now that trade is closed
                if monthly_log and active_combos != monthly_log[-1]['combos']:
                    active_combos = monthly_log[-1]['combos']
                    exit_map      = monthly_log[-1]['exit_map']

                active   = None
                prev_bin = cur_bin
                continue

        # ── Entry ──────────────────────────────────────────────────────────
        if active is None and active_combos:
            long_cross  = (cur_bin in LONG_ENTRY_BINS and
                           cur_bin != prev_bin and
                           prev_bin not in LONG_ENTRY_BINS)
            short_cross = (cur_bin in SHORT_ENTRY_BINS and
                           cur_bin != prev_bin and
                           prev_bin not in SHORT_ENTRY_BINS)

            if long_cross and (cur_bin, vx_bin, 'LONG') in active_combos:
                active = Trade(
                    entry_idx=bar_i, entry_time=ts,
                    entry_price=price,
                    entry_mnq_bin=cur_bin, entry_vx_bin=vx_bin,
                    direction='LONG'
                )
            elif short_cross and (cur_bin, vx_bin, 'SHORT') in active_combos:
                active = Trade(
                    entry_idx=bar_i, entry_time=ts,
                    entry_price=price,
                    entry_mnq_bin=cur_bin, entry_vx_bin=vx_bin,
                    direction='SHORT'
                )

        prev_bin = cur_bin

    # Separate live trades only
    live_completed = [
        t for t in all_trades
        if t.net_pnl_pct is not None and
        t.exit_time is not None and
        t.exit_time >= live_df.index[0]
    ]

    return live_completed, monthly_log

# ── Run ───────────────────────────────────────────────────────────────────────
live_trades, monthly_log = run_walkforward(df)

# ── Performance summary ───────────────────────────────────────────────────────
print(f'\n{"═"*70}')
print(f'  WALK-FORWARD RESULTS — Adaptive Exit + Dual Filter')
print(f'{"═"*70}')

if not live_trades:
    print('No completed trades in live period')
else:
    net_pnls    = [t.net_pnl_pct    for t in live_trades]
    net_dollars = [t.net_pnl_dollar for t in live_trades]
    gross_pnls  = [t.gross_pnl_pct  for t in live_trades]
    holds       = [t.hold_bars       for t in live_trades]

    gross_wr = sum(1 for p in gross_pnls if p > 0) / len(gross_pnls) * 100
    net_wr   = sum(1 for p in net_pnls   if p > 0) / len(net_pnls)   * 100
    total_net= sum(net_dollars)
    tstat, _ = stats.ttest_1samp(net_pnls, 0)

    longs  = [t for t in live_trades if t.direction == 'LONG']
    shorts = [t for t in live_trades if t.direction == 'SHORT']

    print(f'\n  Total trades    : {len(live_trades):,}')
    print(f'    Longs         : {len(longs):,}')
    print(f'    Shorts        : {len(shorts):,}')
    print(f'  Gross WR        : {gross_wr:.1f}%')
    print(f'  Net WR          : {net_wr:.1f}%')
    print(f'  Mean gross PnL  : {np.mean(gross_pnls):+.4f}%')
    print(f'  Mean net PnL    : {np.mean(net_pnls):+.4f}%')
    print(f'  Mean net $/tr   : ${np.mean(net_dollars):+.2f}')
    print(f'  Total net $     : ${total_net:+,.2f}')
    print(f'  Median hold     : {np.median(holds):.0f} bars')
    print(f'  T-stat (net)    : {tstat:+.3f}')

    # Exit bin distribution
    exit_bins_used = defaultdict(int)
    for t in live_trades:
        exit_bins_used[t.exit_bin] += 1
    print(f'\n  Exit bin distribution:')
    for eb, count in sorted(exit_bins_used.items(),
                             key=lambda x: x[1], reverse=True):
        print(f'    {eb:<12} : {count:>5} trades ({count/len(live_trades)*100:.1f}%)')

    # Monthly breakdown
    print(f'\n  {"Month":<10} {"Trades":>7} {"NetWR":>7} '
          f'{"Net$/tr":>9} {"MonthNet$":>11} {"Combos":>7} {"SitOut":>7}')
    print(f'  {"─"*62}')

    for log in monthly_log:
        month_trades = [
            t for t in live_trades
            if t.exit_time is not None and
            t.exit_time.year  == log['date'].year and
            t.exit_time.month == log['date'].month
        ]
        sit = '⚠' if log['n_qualifying'] == 0 else ''
        if not month_trades:
            print(f'  {log["date"].strftime("%Y-%m"):<10} '
                  f'{"—":>7} {"—":>7} {"—":>9} {"—":>11} '
                  f'{log["n_qualifying"]:>7} {sit:>7}')
        else:
            m_nets = [t.net_pnl_dollar for t in month_trades]
            m_wr   = sum(1 for t in month_trades if t.net_pnl_pct > 0)
            print(f'  {log["date"].strftime("%Y-%m"):<10} '
                  f'{len(month_trades):>7} '
                  f'{m_wr/len(month_trades)*100:>6.1f}% '
                  f'{np.mean(m_nets):>+8.2f}$ '
                  f'{sum(m_nets):>+10,.0f}$  '
                  f'{log["n_qualifying"]:>7} {sit:>7}')

    # Sit-out months
    sit_out = [log['date'].strftime('%Y-%m')
               for log in monthly_log if log['n_qualifying'] == 0]
    print(f'\n  Sit-out months ({len(sit_out)}): '
          f'{", ".join(sit_out) if sit_out else "None"}')

    # Combo stability
    combo_months = defaultdict(list)
    for log in monthly_log:
        for key in log['combos']:
            mb, vb, d = key
            opt_exit  = log['exit_map'].get(key, 'fallback')
            combo_months[f'{mb}×{vb} ({d})'].append(
                f'{log["date"].strftime("%Y-%m")}→{opt_exit}'
            )

    print(f'\n  COMBO STABILITY:')
    print(f'  {"Combination":<35} {"Active":>6}  {"Sample months (entry→exit)"}')
    print(f'  {"─"*75}')
    for combo, months in sorted(combo_months.items(),
                                 key=lambda x: len(x[1]), reverse=True):
        sample = ', '.join(months[:4]) + ('...' if len(months) > 4 else '')
        print(f'  {combo:<35} {len(months):>6}x  {sample}')

    # Per-combo breakdown
    print(f'\n  PER-COMBINATION BREAKDOWN:')
    print(f'  {"MNQ Bin":<12} {"VIX Bin":<12} {"Dir":<6} '
          f'{"N":>5} {"NetWR":>7} {"Net$/tr":>9} '
          f'{"TotalNet$":>11} {"MostUsedExit":>13}')
    print(f'  {"─"*78}')

    combo_stats = defaultdict(list)
    for t in live_trades:
        combo_stats[(t.entry_mnq_bin, t.entry_vx_bin,
                     t.direction, t.exit_bin)].append(t)

    # Aggregate by entry combo
    entry_combo_stats = defaultdict(list)
    for t in live_trades:
        entry_combo_stats[(t.entry_mnq_bin, t.entry_vx_bin,
                           t.direction)].append(t)

    rows = []
    for (mb, vb, d), trades in entry_combo_stats.items():
        nets    = [t.net_pnl_pct    for t in trades]
        dollars = [t.net_pnl_dollar for t in trades]
        exits   = [t.exit_bin       for t in trades]
        top_exit= max(set(exits), key=exits.count) if exits else '—'
        rows.append({
            'mb': mb, 'vb': vb, 'd': d,
            'n': len(trades),
            'net_wr': sum(1 for p in nets if p > 0) / len(nets) * 100,
            'net_dollar': np.mean(dollars),
            'total_net':  sum(dollars),
            'top_exit':   top_exit,
        })

    for row in sorted(rows, key=lambda x: x['total_net'], reverse=True):
        sym = '▲' if row['d'] == 'LONG' else '▼'
        print(f'  {row["mb"]:<12} {row["vb"]:<12} {sym}{row["d"]:<4} '
              f'{row["n"]:>5} {row["net_wr"]:>6.1f}% '
              f'{row["net_dollar"]:>+8.2f}$ '
              f'{row["total_net"]:>+10,.0f}$  '
              f'{row["top_exit"]:>13}')

    # ── Charts ────────────────────────────────────────────────────────────
    live_sorted = sorted(live_trades, key=lambda t: t.exit_time)
    cum_net     = np.cumsum([t.net_pnl_dollar for t in live_sorted])
    times       = [t.exit_time for t in live_sorted]

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            'Cumulative Net P&L ($) — Adaptive Exit Walk-Forward',
            'Monthly Active Combinations'
        ],
        vertical_spacing=0.14,
        row_heights=[0.7, 0.3]
    )

    fig.add_trace(go.Scatter(
        x=times, y=cum_net,
        mode='lines', name='Cumulative Net $',
        line=dict(color='#00d4ff', width=1.5)
    ), row=1, col=1)

    fig.add_hline(y=0, line_dash='dash',
                  line_color='#ffffff', opacity=0.3, row=1, col=1)

    for log in monthly_log:
        if log['n_qualifying'] == 0:
            fig.add_vrect(
                x0=log['date'],
                x1=log['date'] + pd.DateOffset(months=1),
                fillcolor='#ff4444', opacity=0.1,
                layer='below', line_width=0
            )

    fig.add_trace(go.Bar(
        x=[log['date'] for log in monthly_log],
        y=[log['n_qualifying'] for log in monthly_log],
        name='Active Combos',
        marker_color=['#ff4444' if log['n_qualifying'] == 0
                      else '#00d4ff' for log in monthly_log],
        opacity=0.8
    ), row=2, col=1)

    fig.update_layout(
        paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
        font=dict(color='#f0f0f0', family='monospace'),
        title=dict(
            text=(f'Adaptive Exit Walk-Forward  |  '
                  f'Monthly Recal  |  {LOOKBACK_DAYS}d lookback  |  '
                  f'WR≥{MIN_NET_WR*100:.0f}% + PnL>0  |  '
                  f'Min n={MIN_TRADES}  |  '
                  f'Cost ${TOTAL_COST_RT:.2f}/rt'),
            font=dict(color='#00d4ff', size=11)
        ),
        height=620,
        margin=dict(l=80, r=80, t=80, b=60),
        showlegend=True,
        legend=dict(bgcolor='#1e2530')
    )
    fig.update_xaxes(gridcolor='#1e2530', color='#8892a4')
    fig.update_yaxes(gridcolor='#1e2530', color='#8892a4')
    fig.update_yaxes(title_text='Cumulative $', row=1, col=1)
    fig.update_yaxes(title_text='# Combos',    row=2, col=1)
    fig.show()

print('\nDone ✓')

# In[8]:


# ── Sharpe & Risk Analysis ────────────────────────────────────────────────────
# Requires live_trades from run_walkforward() to be in scope

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import itertools

completed = sorted(live_trades, key=lambda t: t.exit_time)

if not completed:
    print("No completed trades to analyze")
else:
    exits      = [t.exit_time      for t in completed]
    net_pnls   = [t.net_pnl_dollar for t in completed]
    net_pcts   = [t.net_pnl_pct    for t in completed]
    directions = [t.direction       for t in completed]

    cum_pnl = np.cumsum(net_pnls)

    n_trades   = len(completed)
    total_days = (exits[-1] - exits[0]).days
    years      = total_days / 365.25 if total_days > 0 else 0

    start_capital = 50000.0
    total_net     = sum(net_pnls)

    ending_capital = start_capital + total_net
    ann_return_pct = (
        ((ending_capital / start_capital) ** (1 / years) - 1) * 100
        if years > 0 and ending_capital > 0 else 0
    )

    winners    = [p for p in net_pnls if p > 0]
    losers     = [p for p in net_pnls if p <= 0]
    win_rate   = len(winners) / n_trades * 100
    avg_win    = np.mean(winners) if winners else 0
    avg_loss   = np.mean(losers) if losers else 0
    payoff     = abs(avg_win / avg_loss) if avg_loss != 0 else np.inf
    expectancy = np.mean(net_pnls)

    # ── Realistic daily equity-return Sharpe ──────────────────────────────────
    # Includes zero-P&L business days and computes Sharpe from portfolio returns.
    daily = pd.Series(
        data=[t.net_pnl_dollar for t in completed],
        index=pd.to_datetime([t.exit_time.date() for t in completed])
    )

    daily_pnl_series = daily.groupby(level=0).sum()

    full_days = pd.bdate_range(
        start=daily_pnl_series.index.min(),
        end=daily_pnl_series.index.max()
    )

    daily_pnl_series = daily_pnl_series.reindex(full_days, fill_value=0.0)

    daily_equity = start_capital + daily_pnl_series.cumsum()
    daily_return_series = daily_equity.pct_change().fillna(0)

    daily_pnls = daily_pnl_series.values
    daily_returns = daily_return_series.values

    daily_mean = daily_return_series.mean()
    daily_std  = daily_return_series.std(ddof=1)
    n_days     = len(daily_return_series)

    sharpe = (daily_mean / daily_std) * np.sqrt(252) if daily_std > 0 else 0

    daily_down = daily_return_series[daily_return_series < 0]
    down_std_d = daily_down.std(ddof=1) if len(daily_down) > 1 else 0
    sortino = (daily_mean / down_std_d) * np.sqrt(252) if down_std_d > 0 else 0

    trades_per_year = n_trades / years if years > 0 else 0

    mean_r = np.mean(net_pcts)
    std_r  = np.std(net_pcts, ddof=1)
    sharpe_pertrade = (
        (mean_r / std_r) * np.sqrt(trades_per_year)
        if std_r > 0 and trades_per_year > 0 else 0
    )

    # ── Equity curve drawdown ─────────────────────────────────────────────────
    equity       = start_capital + cum_pnl
    peak         = np.maximum.accumulate(equity)
    drawdown     = equity - peak
    drawdown_pct = drawdown / peak * 100

    max_dd     = drawdown.min()
    max_dd_pct = drawdown_pct.min()

    calmar = ann_return_pct / abs(max_dd_pct) if max_dd_pct != 0 else 0

    tstat, pval = stats.ttest_1samp(net_pnls, 0)

    results = [1 if p > 0 else 0 for p in net_pnls]

    max_win_streak = max(
        (sum(1 for _ in g) for k, g in itertools.groupby(results) if k == 1),
        default=0
    )

    max_loss_streak = max(
        (sum(1 for _ in g) for k, g in itertools.groupby(results) if k == 0),
        default=0
    )

    print(f'\n{"═"*60}')
    print(f'  WALK-FORWARD RISK & PERFORMANCE ANALYSIS')
    print(f'{"═"*60}')

    print(f'\n  Period          : {exits[0].date()} → {exits[-1].date()} ({years:.1f} years)')
    print(f'  Total trades    : {n_trades:,}')
    print(f'  Trades/year     : {trades_per_year:.0f}')

    print(f'\n  ── Returns ──────────────────────────────────────────')
    print(f'  Total net P&L   : ${total_net:+,.2f}')
    print(f'  CAGR            : {ann_return_pct:+.2f}% on ${start_capital:,.0f} capital')
    print(f'  Expectancy      : ${expectancy:+.2f}/trade')

    print(f'\n  ── Win/Loss ─────────────────────────────────────────')
    print(f'  Win rate        : {win_rate:.1f}%')
    print(f'  Avg win         : ${avg_win:+.2f}')
    print(f'  Avg loss        : ${avg_loss:+.2f}')
    print(f'  Payoff ratio    : {payoff:.2f}x')
    print(f'  Max win streak  : {max_win_streak}')
    print(f'  Max loss streak : {max_loss_streak}')

    print(f'\n  ── Risk-Adjusted ────────────────────────────────────')
    print(f'  Sharpe daily    : {sharpe:+.3f}  ← equity-return based')
    print(f'  Sharpe per-tr   : {sharpe_pertrade:+.3f}  ← reference only')
    print(f'  Sortino daily   : {sortino:+.3f}')
    print(f'  Trading days    : {n_days} | Avg daily return: {daily_mean * 100:+.4f}%')
    print(f'  Max drawdown    : ${max_dd:,.2f} ({max_dd_pct:.2f}%)')
    print(f'  Calmar ratio    : {calmar:.3f}')
    print(f'  T-stat          : {tstat:+.3f}  (p={pval:.4f})')
    print(f'  Significant     : {"✓ YES" if pval < 0.05 else "✗ NO"} (p<0.05)')

    print(f'\n  ── Direction Breakdown ──────────────────────────────')

    for d in ['LONG', 'SHORT']:
        d_trades = [t for t in completed if t.direction == d]

        if not d_trades:
            continue

        d_pnls = [t.net_pnl_dollar for t in d_trades]
        d_wr = sum(1 for p in d_pnls if p > 0) / len(d_pnls) * 100
        sym = '▲' if d == 'LONG' else '▼'

        print(
            f'  {sym} {d:<6} : n={len(d_trades):>5}  '
            f'WR={d_wr:.1f}%  '
            f'Total=${sum(d_pnls):+,.2f}  '
            f'Avg=${np.mean(d_pnls):+.2f}'
        )

    print(f'\n  ── Monthly P&L ──────────────────────────────────────')
    print(f'  {"Month":<10} {"Trades":>7} {"WinRate":>8} {"Net$":>10} {"CumNet$":>11}')
    print(f'  {"─"*50}')

    monthly = {}

    for t in completed:
        key = t.exit_time.strftime('%Y-%m')
        monthly.setdefault(key, []).append(t.net_pnl_dollar)

    running_total = 0
    monthly_pnls = []

    for month, pnls in sorted(monthly.items()):
        wr = sum(1 for p in pnls if p > 0) / len(pnls) * 100
        month_net = sum(pnls)
        running_total += month_net
        monthly_pnls.append(month_net)

        print(
            f'  {month:<10} {len(pnls):>7} {wr:>7.1f}% '
            f'{month_net:>+9,.2f}$ {running_total:>+10,.2f}$'
        )

    pos_months = sum(1 for p in monthly_pnls if p > 0)

    print(
        f'\n  Profitable months: {pos_months}/{len(monthly_pnls)} '
        f'({pos_months / len(monthly_pnls) * 100:.1f}%)'
    )

    # ── Charts ────────────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=[
            'Cumulative Net P&L ($)',
            'Drawdown ($)',
            'Rolling 50-Trade Sharpe',
            'Daily Return Distribution (%)',
            'Monthly P&L ($)',
            'Win Rate by Direction',
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )

    fig.add_trace(
        go.Scatter(
            x=exits,
            y=cum_pnl,
            mode='lines',
            name='Cum Net $',
            line=dict(color='#00d4ff', width=1.5)
        ),
        row=1,
        col=1
    )

    fig.add_hline(y=0, line_dash='dash', line_color='#ffffff', opacity=0.3, row=1, col=1)

    fig.add_trace(
        go.Scatter(
            x=exits,
            y=drawdown,
            mode='lines',
            name='Drawdown',
            fill='tozeroy',
            line=dict(color='#ff4444', width=1),
            fillcolor='rgba(255,68,68,0.2)'
        ),
        row=1,
        col=2
    )

    window = 50

    if n_trades > window:
        roll_sharpe = []
        roll_dates = []

        for i in range(window, n_trades):
            w_pcts = net_pcts[i - window:i]
            w_mean = np.mean(w_pcts)
            w_std = np.std(w_pcts, ddof=1)
            rs = (w_mean / w_std) * np.sqrt(trades_per_year) if w_std > 0 else 0

            roll_sharpe.append(rs)
            roll_dates.append(exits[i])

        fig.add_trace(
            go.Scatter(
                x=roll_dates,
                y=roll_sharpe,
                mode='lines',
                name='Rolling Per-Trade Sharpe',
                line=dict(color='#ffaa00', width=1.5)
            ),
            row=2,
            col=1
        )

        fig.add_hline(y=0, line_dash='dash', line_color='#ffffff', opacity=0.3, row=2, col=1)
        fig.add_hline(y=1, line_dash='dot', line_color='#00d4ff', opacity=0.5, row=2, col=1)

    fig.add_trace(
        go.Histogram(
            x=daily_returns * 100,
            nbinsx=60,
            name='Daily Return %',
            marker_color='#00d4ff',
            opacity=0.7
        ),
        row=2,
        col=2
    )

    fig.add_vline(x=0, line_dash='dash', line_color='#ffffff', opacity=0.5, row=2, col=2)

    months_list = sorted(monthly.keys())
    month_nets = [sum(monthly[m]) for m in months_list]

    fig.add_trace(
        go.Bar(
            x=months_list,
            y=month_nets,
            name='Monthly P&L',
            marker_color=['#00d4ff' if p > 0 else '#ff4444' for p in month_nets],
            opacity=0.8
        ),
        row=3,
        col=1
    )

    dir_labels = ['LONG', 'SHORT']
    dir_wrs = []
    dir_counts = []

    for d in dir_labels:
        d_trades = [t for t in completed if t.direction == d]

        if d_trades:
            d_pnls = [t.net_pnl_dollar for t in d_trades]
            dir_wrs.append(sum(1 for p in d_pnls if p > 0) / len(d_pnls) * 100)
            dir_counts.append(len(d_trades))
        else:
            dir_wrs.append(0)
            dir_counts.append(0)

    fig.add_trace(
        go.Bar(
            x=dir_labels,
            y=dir_wrs,
            name='Win Rate %',
            marker_color=['#00d4ff', '#ffaa00'],
            text=[f'{wr:.1f}%<br>n={n}' for wr, n in zip(dir_wrs, dir_counts)],
            textposition='auto',
            opacity=0.8
        ),
        row=3,
        col=2
    )

    fig.add_hline(y=50, line_dash='dash', line_color='#ffffff', opacity=0.3, row=3, col=2)

    fig.update_layout(
        paper_bgcolor='#0d1117',
        plot_bgcolor='#0d1117',
        font=dict(color='#f0f0f0', family='monospace'),
        title=dict(
            text=(
                f'Walk-Forward Risk Analysis | '
                f'Sharpe(daily)={sharpe:.3f} Sortino(daily)={sortino:.3f} '
                f'MaxDD=${max_dd:,.0f} WR={win_rate:.1f}%'
            ),
            font=dict(color='#00d4ff', size=12)
        ),
        height=900,
        margin=dict(l=60, r=60, t=80, b=60),
        showlegend=False
    )

    for row_i in range(1, 4):
        for col_j in range(1, 3):
            fig.update_xaxes(gridcolor='#1e2530', color='#8892a4', row=row_i, col=col_j)
            fig.update_yaxes(gridcolor='#1e2530', color='#8892a4', row=row_i, col=col_j)

    fig.show()

    print('\nDone ✓')

# In[ ]:





Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'your_primary_series_1m.parquet'